# Fine-tuning BLIP — descrição de condição do livro

Antes de rodar:

1. **Add Input** (barra lateral) → anexar os dois Kaggle Datasets:
   - `pedrocanoas/fotos-livros` (fotos)
   - `pedrocanoas/model-livros` (código)
2. **Settings** → Accelerator: **GPU (T4 x2 ou P100)** · Internet: **On**
   (precisa baixar os pesos do BLIP do Hugging Face na primeira execução)

> **Nota sobre o layout do Kaggle:** o Kaggle monta os datasets em
> `/kaggle/input/datasets/<usuário>/<slug>/...` (não mais direto em
> `/kaggle/input/<slug>/...`). Confirme rodando `!find /kaggle/input -maxdepth 4`
> se os caminhos abaixo não baterem com a estrutura do seu notebook.

In [ ]:
# Célula 1 — instala as deps que faltam (não reinstala torch: o Kaggle já vem com build CUDA pronta)
!pip install -q -r /kaggle/input/datasets/pedrocanoas/model-livros/model/requirements-kaggle.txt

In [ ]:
# Célula 2 — copia o código para uma área com permissão de escrita
# Alternativa, se o repo já estiver no GitHub, troque esta célula por:
# !git clone <url-do-repo> /kaggle/working/repo && cp -r /kaggle/working/repo/model /kaggle/working/model
!cp -r /kaggle/input/datasets/pedrocanoas/model-livros/model /kaggle/working/model
%cd /kaggle/working/model

In [ ]:
# Célula 3 — aponta os diretórios de dados/checkpoints para /kaggle/working (única área gravável)
import os

os.environ["TCC_DATASET_RAW_DIR"] = "/kaggle/input/datasets/pedrocanoas/fotos-livros/raw"
os.environ["TCC_DATASET_PROCESSED_DIR"] = "/kaggle/working/processed"
os.environ["TCC_CHECKPOINTS_DIR"] = "/kaggle/working/checkpoints"

In [ ]:
# Célula 4 — gera os manifestos (train.jsonl / val.jsonl) e roda o fine-tuning
# Com 743 imagens numa T4 isso roda em minutos; se faltar VRAM, reduza --batch-size (2 ou 1)
# --max-length 128: com 64 (default anterior), 77% das legendas eram cortadas no treino
# e o modelo nunca aprendia a parar de gerar texto. 128 cobre ~91% das legendas inteiras.
!python kaggle_train.py \
  --raw-dir /kaggle/input/datasets/pedrocanoas/fotos-livros/raw \
  --epochs 20 \
  --batch-size 8 \
  --max-length 128

In [ ]:
# Célula 5 (opcional) — sanity check: gera uma legenda de exemplo com o checkpoint treinado
import glob

sample_photos = sorted(glob.glob("/kaggle/input/datasets/pedrocanoas/fotos-livros/raw/livro001/foto*.jpg"))[:1]
if sample_photos:
    !python -m src.infer {sample_photos[0]}
else:
    print("ajuste o caminho de exemplo acima para uma foto existente no seu dataset")

In [ ]:
# Célula 6 — empacota o melhor checkpoint para trazer de volta
!zip -r /kaggle/working/checkpoint_best.zip /kaggle/working/checkpoints/best

Depois de rodar tudo, clique em **Save Version** (Save & Run All) para commitar o
notebook — o `checkpoint_best.zip` aparece na aba **Output** para baixar direto
pelo navegador, ou via Kaggle API:

```sh
kaggle kernels output <seu-usuario>/<slug-do-notebook> -p ./kaggle-output
```

Extraia o zip de forma que o resultado fique em `model/checkpoints/best/`
(caminho que `infer.py` usa por padrão localmente).